In [1]:
!pip install -U datasets fsspec transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency

In [4]:
import warnings
warnings.filterwarnings("ignore",category=FutureWarning)

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler
)
from tqdm.auto import tqdm


In [35]:
# --------------------------------------------------
# 1️⃣ Hyperparameters
# --------------------------------------------------

batch_size = 16
lr = 5e-5
epochs = 1
temperature = 2.0
alpha = 0.5
max_len = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [36]:
# --------------------------------------------------
# 2️⃣ Load Dataset
# --------------------------------------------------

raw = load_dataset("tweet_eval", "sentiment")
num_labels = raw["train"].features["label"].num_classes

# Small subset for demo (remove .select() for full dataset)
train_ds = raw["train"].shuffle(seed=42).select(range(2500))
val_ds = raw["validation"]

In [37]:
# --------------------------------------------------
# 3️⃣ Tokenization
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=max_len
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

train_ds.set_format("torch")
val_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer)

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)

val_dl = DataLoader(
    val_ds,
    batch_size=batch_size,
    collate_fn=data_collator
)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [38]:
# --------------------------------------------------
# 4️⃣ Teacher Model (BERT-Large)
# --------------------------------------------------

teacher = AutoModelForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=num_labels
).to(device)

# Freeze teacher
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-large-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
# --------------------------------------------------
# 5️⃣ Student Model (BERT-Base)
# --------------------------------------------------

student = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
# --------------------------------------------------
# 6️⃣ Optimizer & Scheduler
# --------------------------------------------------

optimizer = optim.AdamW(student.parameters(), lr=lr)

num_training_steps = len(train_dl) * epochs

scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction="batchmean")

In [41]:
# --------------------------------------------------
# 7️⃣ Distillation Training Loop
# --------------------------------------------------

progress_bar = tqdm(range(num_training_steps))

for epoch in range(epochs):
    student.train()
    total_loss = 0

    for batch in train_dl:

        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]

        # -----------------------------
        # Teacher forward (no grad)
        # -----------------------------
        with torch.no_grad():
            teacher_outputs = teacher(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            teacher_logits = teacher_outputs.logits
            teacher_probs = torch.softmax(
                teacher_logits / temperature, dim=-1
            )

        # -----------------------------
        # Student forward
        # -----------------------------
        student_outputs = student(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        student_logits = student_outputs.logits
        student_log_probs = torch.log_softmax(
            student_logits / temperature, dim=-1
        )

        # -----------------------------
        # Soft Loss (KL)
        # -----------------------------
        loss_soft = kl_loss(student_log_probs, teacher_probs) * (temperature ** 2)

        # -----------------------------
        # Hard Loss (CE)
        # -----------------------------
        loss_hard = ce_loss(student_logits, labels)

        # -----------------------------
        # Combined Loss
        # -----------------------------
        loss = alpha * loss_soft + (1 - alpha) * loss_hard

        # -----------------------------
        # Backpropagation
        # -----------------------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.update(1)

    print(f"\nEpoch {epoch+1} Loss: {total_loss/len(train_dl):.4f}")

  0%|          | 0/157 [00:00<?, ?it/s]


Epoch 1 Loss: 0.5385


In [42]:
# --------------------------------------------------
# 8️⃣ Evaluation
# --------------------------------------------------

student.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in val_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = student(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

print(f"\nValidation Accuracy: {correct/total:.4f}")


Validation Accuracy: 0.6350


In [43]:
student.save_pretrained("distilled_student_model")
tokenizer.save_pretrained("distilled_student_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilled_student_model/tokenizer_config.json',
 'distilled_student_model/tokenizer.json')

In [44]:
test_ds = raw["test"]

test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])
test_ds = test_ds.rename_column("label", "labels")
test_ds.set_format("torch")

test_dl = DataLoader(
    test_ds,
    batch_size=batch_size,
    collate_fn=data_collator
)

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

In [45]:
student.eval()

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dl:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = student(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

test_accuracy = correct / total
print(f"\nTest Accuracy: {test_accuracy:.4f}")



Test Accuracy: 0.5960


In [46]:
id2label = raw["train"].features["label"].names

for i in range(5):
    print(f"\nExample {i+1}")
    print("Predicted:", id2label[all_preds[i]])
    print("Actual   :", id2label[all_labels[i]])


Example 1
Predicted: neutral
Actual   : neutral

Example 2
Predicted: neutral
Actual   : neutral

Example 3
Predicted: neutral
Actual   : neutral

Example 4
Predicted: neutral
Actual   : positive

Example 5
Predicted: neutral
Actual   : negative


In [47]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def evaluate_model(model, dataloader, name="Model"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"\n{name} Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")

    return acc, f1


In [48]:
print("\nEvaluating Teacher...")
teacher_acc, teacher_f1 = evaluate_model(teacher, test_dl, name="Teacher")

print("\nEvaluating Student...")
student_acc, student_f1 = evaluate_model(student, test_dl, name="Student")



Evaluating Teacher...

Teacher Results:
Accuracy: 0.3332
Macro F1: 0.1918

Evaluating Student...

Student Results:
Accuracy: 0.5960
Macro F1: 0.5439
